# 10 · Coming from Pytorch-Wildlife

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microsoft/SPARROW-Playground/blob/main/notebooks/10_from_pytorch_wildlife.ipynb)

SPARROW-Playground replaces Pytorch-Wildlife. The models are the same; they now run on
Sparrow-Engine (ONNX) instead of PyTorch. This notebook maps the old API to the new one.
Fine-tuning moved to [MegaDetector-Classifier](https://github.com/microsoft/MegaDetector-Classifier).

## Model names

| Pytorch-Wildlife | Sparrow-Engine model id |
|---|---|
| `pw_detection.MegaDetectorV6(version="MDV6-yolov9-c")` | `MDV6-yolov9-c` |
| `pw_detection.MegaDetectorV6(version="MDV6-yolov9-e")` | `MDV6-yolov9-e` |
| `pw_detection.MegaDetectorV6(version="MDV6-yolov10-c")` | `MDV6-yolov10-c` |
| `pw_detection.MegaDetectorV6(version="MDV6-yolov10-e")` | `MDV6-yolov10-e` |
| `pw_detection.MegaDetectorV6(version="MDV6-rtdetr-c")` | `MDV6-rtdetr-c` |
| `pw_detection.MegaDetectorV6MIT(...)` | `MDV6-mit-yolov9-c`, `MDV6-mit-yolov9-e` |
| `pw_detection.MegaDetectorV6Apache(...)` | `MDV6-apa-rtdetr-c`, `MDV6-apa-rtdetr-e` |
| `pw_detection.DeepfauneDetector()` | `deepfaune-yolo8s` |
| `pw_detection.HerdNet()` | `HerdNet_General_Dataset_2022` |
| `pw_detection.OWLC()` / `OWLT()` | `OWL` |
| `pw_classification.AI4GAmazonRainforest()` | `AI4G-Amazon-V2` |
| `pw_classification.AI4GSnapshotSerengeti()` | `AI4G-Serengeti` |
| `pw_classification.DeepfauneClassifier()` | `Deepfaune-Europe` |
| `pw_classification.DFNE()` | `Deepfaune-New-England` |
| `pw_detection.MegaDetectorV5()` | `MDV5a` (listed, not yet downloadable) |
| `pw_classification.AI4GOpossum()` | not in the zoo |

In [ ]:
# Setup: install sparrow-engine if it is missing (for example on Colab).
# For an NVIDIA GPU, install `sparrow-engine-gpu` instead (same Python API).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("sparrow_engine") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sparrow-engine[video]>=0.1.31"])

import sparrow_engine as spe

print("sparrow-engine", spe.__version__, "| device:", spe.active_device())

In [ ]:
# Demo data: downloaded once into ./data and checked against a published checksum.
import hashlib
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path

DATA = Path(os.environ.get("SPP_DATA_DIR", "data")).resolve()
ASSETS = {
    # Pytorch-Wildlife demo assets, CC-BY-4.0, https://zenodo.org/records/15376499
    "demo_data_det.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_det.zip?download=1",
        "md5", "373e15c8e7c8894379a82086a9177763",
    ),
    "demo_data_base.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_base.zip?download=1",
        "md5", "f89b9b76c753d90e4296d62d71cf1732",
    ),
    # "Yellow-Throated Vireo Song" by SwampVids, CC-BY-4.0, Wikimedia Commons
    "vireo.wav": (
        "https://upload.wikimedia.org/wikipedia/commons/4/4f/"
        "Vireo_flavifrons_-_Yellow-Throated_Vireo_Song.wav",
        "sha1", "2b472209c1b18bfc0c225975838e3a34c1e9887a",
    ),
}


def fetch(name):
    """Download an asset (once), verify it, and unzip archives into DATA."""
    url, algo, digest = ASSETS[name]
    target = DATA / name
    DATA.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        print(f"downloading {name} ...")
        part = target.with_name(target.name + ".part")
        req = urllib.request.Request(url, headers={"User-Agent": "sparrow-playground"})
        with urllib.request.urlopen(req) as r, open(part, "wb") as f:
            shutil.copyfileobj(r, f)
        part.rename(target)
    h = hashlib.new(algo)
    with open(target, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    if h.hexdigest() != digest:
        target.unlink()
        raise RuntimeError(f"{name}: checksum mismatch, deleted; run the cell again")
    marker = DATA / f".{name}.extracted"
    if name.endswith(".zip") and not marker.exists():
        with zipfile.ZipFile(target) as z:
            z.extractall(DATA)
        marker.touch()
    return target

In [ ]:
fetch("demo_data_det.zip")
fetch("demo_data_base.zip")  # about 95 MB
folder = DATA / "data/data_example/images/val"
image = DATA / "demo_data/imgs/10050028_1.JPG"

In [ ]:
# Models are installed into SPARROW_ENGINE_MODEL_DIR (default ~/.sparrow-engine/models).
# Already-installed models are skipped. Inference never downloads anything by itself.
for model_id in ["MDV6-yolov10-c", "AI4G-Amazon-V2"]:
    spe.download_model(model_id)

## Single image

Before:
```python
from PytorchWildlife.models import detection as pw_detection
model = pw_detection.MegaDetectorV6(device="cuda", pretrained=True, version="MDV6-yolov10-c")
result = model.single_image_detection(img_path, det_conf_thres=0.2)
result["detections"].xyxy, result["detections"].confidence
```
After:

In [ ]:
result = spe.detect(image, model="MDV6-yolov10-c", threshold=0.2)[0]
[(d.label, d.confidence, d.bbox) for d in result.detections]

Differences: there is no `device` argument (the engine picks the GPU when the
`sparrow-engine-gpu` package is installed; `spe.init(device="cpu")` forces CPU),
and boxes are normalized to `[0, 1]` instead of pixels. Multiply by `result.image_size`
(width, height) for pixels.

## Folder

Before:
```python
results = model.batch_image_detection(folder, batch_size=16, det_conf_thres=0.2)
```
After:

In [ ]:
results = spe.detect(folder, model="MDV6-yolov10-c", threshold=0.2)
len(results)

## Saving results

Before:
```python
from PytorchWildlife import utils as pw_utils
pw_utils.save_detection_images(results, "out_images", folder)
pw_utils.save_detection_json(results, "out.json", categories=model.CLASS_NAMES)
```
After:

In [ ]:
out = Path("outputs/10_migration")
spe.visualize(results, output_dir=out / "images")
spe.export(results, "megadet", out / "out.json", model_id="MDV6-yolov10-c")
sorted(p.name for p in out.iterdir())

## Detection + classification

Before:
```python
detector = pw_detection.MegaDetectorV6(version="MDV6-yolov10-c")
classifier = pw_classification.AI4GAmazonRainforest()
det = detector.single_image_detection(img_path)
# crop each box with supervision, then classifier.single_image_classification(crop)
```
After, one call:

In [ ]:
r = spe.pipeline(image, detector="MDV6-yolov10-c", classifier="AI4G-Amazon-V2")[0]
[(d.detection.label, d.classification.label, round(d.classification.confidence, 2))
 for d in r.detections]

## What is gone

| Pytorch-Wildlife | Now |
|---|---|
| Fine-tuning code | [MegaDetector-Classifier](https://github.com/microsoft/MegaDetector-Classifier) |
| Gradio demo | These notebooks |
| `pip install PytorchWildlife` | `pip install sparrow-engine` |
| PyTorch, torchvision, ultralytics | Not needed |